# 访问华中杯官网首页

## 定义访问地址

In [ ]:
url='http://hzbmmc.com:8080/jeecg-boot/api/v1/webbaseinfo/announcement/brief?mentTypehzbWebbaseinfoAnnounce=1&queryPage=1&querySize=5'

## 发送请求

In [ ]:
import requests

In [ ]:
response=requests.get(url)

In [ ]:
response.status_code

In [ ]:
response.text

## 解析数据

动态页面的数据解析和静态页面不太一样，动态数据的响应内容是JSON格式数据，不能使用xpath表达式进行数据解析

In [ ]:
#将json数据转化为字典格式
res_json = response.json()

In [ ]:
res_json

In [ ]:
title=[i['title'] for i in res_json['list']]

In [ ]:
time=[i['releaseTime'] for i in res_json['list']]

## 保存数据

In [ ]:
import pandas as pd

In [ ]:
dataframe=pd.DataFrame([title,time],index=['标题','时间'])
dataframe

In [ ]:
dataframe.to_excel('华中杯标题时间.xlsx')

# 访问华中杯的通知与公告

## 批量获取所有通知与公告标题内容

In [ ]:
start_url='http://hzbmmc.com:8080/jeecg-boot/api/v1/webbaseinfo/announcement/brief?keyWord=&queryPage=1&querySize=4&hzbWebbaseinfoAnnouncementType='

In [ ]:
#获取最大页面数
max_page=requests.get(start_url).json()['totalPage']
max_page

In [ ]:
titles=[]
for i in range(1,max_page+1):
    #获取网页地址
    url=f'http://hzbmmc.com:8080/jeecg-boot/api/v1/webbaseinfo/announcement/brief?keyWord=&queryPage={i}&querySize=4&hzbWebbaseinfoAnnouncementType='
    #发送请求
    response=requests.get(url)
    #将响应的JSON数据转化为字典
    res_json = response.json()
    #数据解析
    title=[i['title'] for i in res_json['list']]
    titles.extend(title)
    print(f'{i}/{max_page}')
titles

In [ ]:
#数据储存
with open('华中杯通知.text','w',encoding='utf-8')as f:
    for  i in titles:
        num=titles.index(i)
        f.write(str(num+1)+':'+i+'\n')

# 热门微博页面数据抓取

## 定义网址

In [ ]:
url='https://weibo.com/ajax/feed/hottimeline?refresh=2&group_id=102803&containerid=102803&extparam=discover|new_feed&max_id=1&count=10'

In [ ]:
res=requests.get(url)
res.status_code

In [ ]:
res.text

## 数据解析

In [ ]:
 res_json = res.json()

In [ ]:
res_json

### 获取博文文本

In [ ]:
text=[i['text_raw'].replace('\u200b','')for i in res_json['statuses']]
text

### 获取博文发布时间

In [ ]:
import time

In [ ]:
c=[i['created_at'].replace('\u200b','')for i in res_json['statuses']]
c

In [ ]:
time=[time.strftime(' %H',time.strptime(i, '%a %b %d %H:%M:%S %z %Y')) for i in c]
time

### 获取转发数

In [ ]:
reposts_count=[i['reposts_count']for i in res_json['statuses']]
reposts_count

### 获取评论数

In [ ]:
comments_count=[i['comments_count']for i in res_json['statuses']]
comments_count

### 获取点赞数

In [ ]:
attitudes_count=[i['attitudes_count']for i in res_json['statuses']]
attitudes_count

### 获取用户昵称

In [ ]:
screen_name=[i['user']['screen_name'] for i in res_json['statuses']]

In [ ]:
screen_name

## 翻页抓取

In [ ]:
#思考
max_page = 10

In [ ]:
all_screen_name=[]
all_text=[]
all_time=[]
all_reposts_count=[]
all_comments_count=[]
all_attitudes_count=[]
for page in range(max_page):
    url=f'''https://weibo.com/ajax/feed/hottimeline?refresh=2&group_id=102803&
    containerid=102803&extparam=discover|new_feed&max_id={page}&count=10'''
    res=requests.get(url)
     
    text=[i['text_raw'].replace('\u200b','')for i in res_json['statuses']]
    time=[i['created_at'].replace('\u200b','')for i in res_json['statuses']]
    reposts_count=[i['reposts_count']for i in res_json['statuses']]
    comments_count=[i['comments_count']for i in res_json['statuses']]
    attitudes_count=[i['attitudes_count']for i in res_json['statuses']]
    screen_name=[i['user']['screen_name'] for i in res_json['statuses']]
    
    all_screen_name.extend(screen_name)
    all_text.extend(text)
    all_time.extend(time)
    all_reposts_count.extend(reposts_count)
    all_comments_count.extend(comments_count)
    all_attitudes_count.extend(attitudes_count)
    print(f'{page+1}/{max_page}----{len(all_screen_name),len(all_text),len(all_time),len(all_reposts_count),len(all_comments_count),len(all_attitudes_count)}')

## 数据储存

In [ ]:
data=pd.DataFrame([all_screen_name,all_text,all_time,all_reposts_count,all_comments_count,all_attitudes_count]
                  ,index=['用户昵称','博文文本','博文发布时间','转发数','评论数','点赞数']).T
data

In [ ]:
data.to_excel('热门微博.xlsx')

# 首页博文数据抓取

## 定义初始网址

In [ ]:
start_url=''